# Manual CAT Sessions with CatEngine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/douglasrizzo/catsim/blob/main/notebooks/02_manual_cat_sessions.ipynb)

Use `CatEngine` when your software owns the CAT loop. This is the right
workflow for applications with a UI, API, or database that receives real
responses from examinees.

In [ ]:
import dataclasses

import numpy as np

from catsim import Estimate
from catsim.engine import CatEngine, RunContext
from catsim.estimation import NumericalSearchEstimator
from catsim.initialization import FixedPointInitializer
from catsim.item_bank import ItemBank
from catsim.selection import MaxInfoSelector
from catsim.state import CatSessionState, ExposureTracker
from catsim.stopping import MaxLengthStopper, StandardErrorStopper, StopRule

## Create the item bank and engine

The stopper is a `StopRule` that combines a precision rule with a length cap:
the session ends once the standard error of the ability estimate is at most
0.5, or at 20 items, and never before 5 items.

In [ ]:
item_bank = ItemBank.generate_item_bank(80, seed=11)
engine = CatEngine(
  item_bank=item_bank,
  initializer=FixedPointInitializer(0.0),
  selector=MaxInfoSelector(r_max=0.5),
  estimator=NumericalSearchEstimator(),
  stopper=StopRule(StandardErrorStopper(0.5), MaxLengthStopper(20), min_items=5),
)
tracker = ExposureTracker(item_bank.n_items)
context = RunContext(rng=np.random.default_rng(2024), exposure_tracker=tracker)

## Start a session

A `CatSessionState` is the runtime object you would typically persist or
serialize in an external system.

In [ ]:
session = engine.start_session(
  session_id=501,
  context=context,
  true_theta=None,
)

print(session)
print("Theta history:", session.theta_history)

## Select and apply items step by step

`select_next` returns the next item to administer, or `None` once the session
has ended, so it is the only stop check the loop needs. `apply_response`
records the response, updates the ability estimate and asks the stopper
whether the session is over.

In a real system, the response would come from an examinee. Here a short list
of mocked responses, repeated from the start when it runs out, keeps the
example deterministic. Each line also prints `se`, the standard error the
estimator reported with the updated estimate, which is the value
`StandardErrorStopper` compares with its target.

In [ ]:
response_stream = [True, True, False, True, False, True, True]

while (next_item := engine.select_next(session, context)) is not None:
  response = response_stream[len(session.responses) % len(response_stream)]
  step = engine.apply_response(session, next_item, response, context)
  print({
    "item": step.selected_item_id,
    "response": step.response,
    "updated_theta": round(step.updated_theta, 3),
    "se": round(step.session.current_se, 3),
    "stopped": step.stopped,
  })

## Read the session state

The application can inspect the updated state after every response. This
session stopped because `se` fell to 0.5 or below before the 20-item cap, so its
stop reason is `stopper`: the stopping rule ended it. A session whose selector
has no item left to offer ends with `no_items` instead. Both values are members
of `catsim.StopReason`. While a session runs, its stop reason is `None` and
`is_stopped` is false.

In [ ]:
print("Administered items:", session.administered_item_ids)
print("Responses:", session.responses)
print("Theta history:", [round(theta, 3) for theta in session.theta_history])
print("Stopped:", session.is_stopped)
print("Stop reason:", session.stop_reason)

## Reconstruct a session

If you persist the session fields in another system, you can reconstruct a
`CatSessionState` later and continue from there. Its `estimates` hold the
initial estimate and one `Estimate` per response, each with its `theta`, its
standard error `se` and the estimator's `evaluations`; the current estimate is
the last one, and `theta_history` is read from them. Store the three fields of
each estimate and rebuild the `Estimate`s when restoring: the state rejects
anything else, and a list that does not match its responses.

In [ ]:
# What a database row would hold: plain values rather than catsim objects.
stored_estimates = [dataclasses.asdict(estimate) for estimate in session.estimates]
print(stored_estimates[:2])

restored = CatSessionState(
  session_id=session.session_id,
  estimates=[Estimate(**fields) for fields in stored_estimates],
  true_theta=session.true_theta,
  administered_item_ids=list(session.administered_item_ids),
  responses=list(session.responses),
  stop_reason=session.stop_reason,
)

print(restored.administered_count)
print(round(restored.current_theta, 3), round(restored.current_se, 3))
print(restored == session)

## Cap item exposure across sessions

An exposure cap limits how often one item is given across sessions:
`MaxInfoSelector(r_max=0.5)` passes over an item whose exposure rate has
reached 0.5, unless no other item remains. The rates need state that outlives
a session, the `ExposureTracker` in the run context, shared by every session of
the deployment. The engine counts each session it starts and each item it
administers, and an item's rate is the share of the sessions started before the
current one that received it. Session 501 was the first, so it saw every rate
at 0. A second session sees each of 501's items at a rate of 1, so it starts
with another item although its first estimate is the same. Without a tracker,
the selector gets no rates and applies no cap.

In [ ]:
second = engine.start_session(session_id=502, context=context)
first_item = session.administered_item_ids[0]

print(f"Rate of item {first_item} seen by session 502:", tracker.rates()[first_item])
print("First item of session 501:", first_item)
print("First item of session 502:", engine.select_next(second, context))

## Integration notes

In a production system, the calling application should usually persist at
least:

- `session_id`, any hashable value, such as an int, a str or a UUID
- `estimates`: each estimate's `theta`, `se` and `evaluations`
- `administered_item_ids`
- `responses`
- `stop_reason`

Data that ties the CAT session to a user or attempt belongs to the application,
stored next to these fields.

The exposure tracker is shared by every session, so it is saved apart from
them: keep its `counts` and `n_sessions`, and after a restart rebuild it with
`ExposureTracker(item_bank.n_items, counts=..., n_sessions=...)`. A fresh tracker
would apply no exposure cap until new sessions had run.

The application layer should remain responsible for authentication, UI,
scoring policies outside IRT, and storage. `catsim` should own the CAT
decision logic itself.